# Retrain "Hey Sebastian" — 4-syllable pronunciationRebuilt from the original openWakeWord auto-training notebook with everycompatibility fix from the first run already applied, so this should runtop to bottom without the dependency archaeology that took a few hours last time.**How to run:** Runtime → Change runtime type → **T4 GPU**, then run cells in order.Stop at the *Pronunciation preview* section and actually listen before training —that step is the whole point of this run.Total time: roughly 45-60 min, most of it the dataset download and Step 1.

## 1. Environment setupInstalls everything and applies all known compatibility patches.

In [ ]:
# Repos + packages!git clone -q https://github.com/rhasspy/piper-sample-generator# Pinned deliberately. Later commits moved generate_samples.py into a package# (breaking train.py's import) and rewrote it against a Piper release whose# piper-phonemize dependency has no Linux wheel for Colab's Python 3.12.!cd piper-sample-generator && git checkout -q ded9350!mkdir -p piper-sample-generator/models!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt!git clone -q https://github.com/dscripka/openwakeword!pip install -q -e ./openwakeword!pip install -q piper-tts webrtcvad onnxscript soundfile!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14!pip install -q audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6!pip install -q pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19# Feature-extraction models, which openwakeword expects at a fixed pathimport osos.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)!wget -q -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx!wget -q -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnxprint("environment ready")

In [ ]:
# --- Compatibility patches (all discovered the hard way on the first run) ---import re, importlib.util, pathlib# 1. torch_audiomentations calls two torchaudio APIs that no longer exist in#    the version Colab ships. Patch the installed library in place, since it#    is imported inside a subprocess where a monkeypatch here wouldn't reach.io_path = pathlib.Path(importlib.util.find_spec("torch_audiomentations").origin).parent / "utils" / "io.py"src = io_path.read_text()src = re.sub(r"^(\s*)torchaudio\.set_audio_backend.*$",             r"\1pass  # patched: removed in newer torchaudio",             src, flags=re.M)if "_torchaudio_info_shim" not in src:    shim = (        "\nimport soundfile as _sf"        "\nfrom types import SimpleNamespace as _SimpleNamespace"        "\ndef _torchaudio_info_shim(filepath, **kwargs):"        "\n    _i = _sf.info(filepath)"        "\n    return _SimpleNamespace(sample_rate=_i.samplerate, num_frames=_i.frames, num_channels=_i.channels)"        "\ntorchaudio.info = _torchaudio_info_shim\n"    )    src = src.replace("import torchaudio", "import torchaudio" + shim, 1)io_path.write_text(src)print(f"patched {io_path}")# 2. train.py calls generate_samples() without a model argument, but this#    pinned revision has no default for it.gs = pathlib.Path("piper-sample-generator/generate_samples.py")s = gs.read_text().replace(    "model: Union[str, Path],",    'model: Union[str, Path] = "piper-sample-generator/models/en_US-libritts_r-medium.pt",',    1)gs.write_text(s)print("patched generate_samples.py default model")# 3. train.py imports generate_samples as a top-level moduleos.environ["PYTHONPATH"] = os.path.abspath("piper-sample-generator") + os.pathsep + os.environ.get("PYTHONPATH", "")print("PYTHONPATH set")

In [ ]:
import os, sys, numpy as np, torch, uuid, yaml, datasets, scipyfrom pathlib import Pathfrom tqdm import tqdmprint("imports ok")

## 2. Pronunciation preview — listen before trainingGenerates one clip per candidate spelling. **Play each one** and note which sound like *your* pronunciation (suh-BAS-tee-uhn). You'll pick the winners in the next cell.Piper pronounces from spelling, so alternate spellings are how we steer it toward 4 syllables.

In [ ]:
import syssys.path.insert(0, os.path.abspath("piper-sample-generator"))from generate_samples import generate_samplesfrom IPython.display import Audio, displayimport glob, shutilCANDIDATES = [    "hey sebastian",     # baseline — the 3-syllable one already trained    "hey sebastien",    "hey sebastiaan",    "hey sebasteean",    "hey sebastee an",    "hey se bas tee an",    "hey sebastyan",    "hey sebasteon",]shutil.rmtree("preview", ignore_errors=True)for i, phrase in enumerate(CANDIDATES):    d = f"preview/{i}"    os.makedirs(d, exist_ok=True)    generate_samples(text=[phrase], max_samples=1, batch_size=1,                     noise_scales=[0.6], noise_scale_ws=[0.6], length_scales=[1.0],                     output_dir=d, file_names=[f"{i}.wav"])for i, phrase in enumerate(CANDIDATES):    f = glob.glob(f"preview/{i}/*.wav")    if f:        print(f"[{i}] {phrase}")        display(Audio(f[0]))

### Pick your spellingsEdit the list below to just the ones that sounded right. Keeping `hey sebastian` too means the model answers to **both** pronunciations — recommended, since you've already adapted to the 3-syllable one.

In [ ]:
# EDIT THIS — keep only the spellings that sounded like your pronunciation.TARGET_PHRASES = [    "hey sebastian",    "hey sebastien",    "hey sebasteean",]print(TARGET_PHRASES)

## 3. Download training data~20 min, mostly the 16GB precomputed feature file. A 404 on the AudioSet tar is expected and harmless — that mirror moved, and FMA below covers background noise.

In [ ]:
# Room impulse responses (MIT)output_dir = "./mit_rirs"if not os.path.exists(output_dir):    os.mkdir(output_dir)    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)    for row in tqdm(rir_dataset):        name = row['audio']['path'].split('/')[-1]        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))print("rirs done")

In [ ]:
# Background noise/music (FMA). AudioSet is skipped — its mirror 404s now.output_dir = "./fma"if not os.path.exists(output_dir):    os.mkdir(output_dir)    fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)    fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))    n_hours = 1    for i in tqdm(range(n_hours*3600//30)):        row = next(fma_dataset)        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))print("fma done")

In [ ]:
# Precomputed openWakeWord features (~16GB training + validation set)if not os.path.exists("openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):    !wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npyif not os.path.exists("validation_set_features.npy"):    !wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npyprint("features done")

## 4. Training config

In [ ]:
# Start from the repo's own default config so every key it expects is present,# then override only what this run changes.config = yaml.load(open("openwakeword/examples/custom_model.yml").read(), yaml.Loader)config["target_phrase"] = TARGET_PHRASESconfig["model_name"] = "hey_sebastian_v2"# 3x the notebook default: the sample budget is split across the spellings# above, so this keeps roughly the same number of clips per pronunciation# as the first (working) model got for its single one.config["n_samples"] = 3000config["n_samples_val"] = 1000config["steps"] = 10000# Left at the notebook defaults — these are early-stopping targets, and the# first model cleared them comfortably (0.72 accuracy / 0.44 recall).config["target_accuracy"] = 0.6config["target_recall"] = 0.25# AudioSet is dropped: its mirror 404s now. FMA alone covers background noise.config["background_paths"] = ["./fma"]config["background_paths_duplication_rate"] = [1]config["false_positive_validation_data_path"] = "validation_set_features.npy"config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}with open("my_model.yaml", "w") as f:    yaml.dump(config, f)print(yaml.dump(config))

## 5. TrainRun these in order. Step 1 is the slow one (~15-20 min).

In [ ]:
# Step 1: generate synthetic clips!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

In [ ]:
# Resample to 16kHz — this pinned generate_samples writes at the model's# native rate with no resampling step, and augmentation rejects anything# that isn't 16k ("Clip does not have the correct sample rate!").import soundfile as sffrom scipy.signal import resample_polyimport globdirs = [os.path.join(config["output_dir"], config["model_name"], d)        for d in ["positive_train", "positive_test", "negative_train", "negative_test"]]fixed = 0for d in dirs:    if not os.path.isdir(d):        continue    for p in glob.glob(os.path.join(d, "*.wav")):        data, sr = sf.read(p)        if sr != 16000:            sf.write(p, resample_poly(data, 16000, sr).astype(np.float32), 16000, subtype="PCM_16")            fixed += 1print(f"resampled {fixed} files to 16kHz")

In [ ]:
# Step 2: augment clips + compute features!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# Step 3: train the classifier and export ONNX.# The tflite conversion at the very end will fail on onnx_tf/py3.12 — that is# expected and harmless, the .onnx is already written by then.!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

## 6. Download the modelGrab **both** files — the classifier stores its weights externally, so the `.onnx` alone is incomplete.

In [ ]:
from google.colab import filesimport glob, osout = glob.glob("my_custom_model/*.onnx*")print("found:", out)for f in out:    print(f, os.path.getsize(f), "bytes")    files.download(f)